<a href="https://colab.research.google.com/github/HC-AIGroup/CS5330_F26/blob/main/Module%206/Segmentation_Implementation_module_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentation in Practice: Pretrained Models & Transfer Learning
### CS 5330 · Colab notebook

| Part | What it does |
|---|---|
| **Setup** | GPU check, installs, helpers |
| **Demo 1** | Pretrained **DeepLabV3** on your own photos (no training) |
| **Data** | Oxford-IIIT Pet: images + trimaps → binary pet masks |
| **Exp 1** | FCN-ResNet50, **pretrained backbone frozen**, train the head |
| **Exp 2** | U-Net **from scratch** (nothing pretrained) |
| **Exp 3** | U-Net + ResNet34 encoder, **encoder frozen** |
| **Exp 4** | U-Net + ResNet34, **fine-tune all** (smaller LR for the encoder) |
| **Compare** | One table, one chart, same images side by side |
| **Debug lesson** | What happens when the backbone is not really pretrained |

**How to run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all* (about 10–15 minutes; downloading the pet dataset is the slowest part).
Every experiment uses the **same** images, so the comparison is fair.

**Try it yourself:** change `EPOCHS`, `N_TRAIN`, the learning rates, or `encoder_name` (e.g. `"efficientnet-b0"`, `"mobilenet_v2"`) and see how the results change.

---
## Setup

In [ ]:
!pip -q install segmentation-models-pytorch
import os, time, random, json, io
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torchvision.transforms import functional as TF
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
if device.type == "cpu":
    print("⚠️  No GPU: Runtime → Change runtime type → T4 GPU, then run again.")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]     # ImageNet statistics
def denormalize(x):
    m = torch.tensor(MEAN, device=x.device).view(3, 1, 1); s = torch.tensor(STD, device=x.device).view(3, 1, 1)
    return (x * s + m).clamp(0, 1)

def logits_of(model, x):
    """torchvision segmentation models return a dict; smp and our U-Net return a tensor."""
    out = model(x)
    return out["out"] if isinstance(out, dict) else out

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total

os.makedirs("results", exist_ok=True)
RESULTS = {}     # final numbers of each experiment

---
## Demo 1 · Pretrained DeepLabV3 (no training)
DeepLabV3 with a ResNet-50 backbone, trained on COCO images but predicting the **21 Pascal VOC classes**.

**Model = architecture + weights.** torchvision builds the architecture, then downloads the weights (`.pth` file) once and caches them.

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

weights = DeepLabV3_ResNet50_Weights.DEFAULT          # Step 2: pretrained weights
dl_model = deeplabv3_resnet50(weights=weights).to(device).eval()
preprocess = weights.transforms()                     # Step 3: the preprocessing it was trained with
categories = weights.meta["categories"]               # Step 6: class names
print("Weights cached in:", os.path.join(torch.hub.get_dir(), "checkpoints"))
print(len(categories), "classes:", categories)

In [ ]:
# Step 1: load images. Upload your own (you can select several files at once).
# If you upload nothing, three sample images are used.
images = {}
try:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        images[name] = Image.open(io.BytesIO(data)).convert("RGB")
except Exception as e:
    print("No upload:", e)
if not images:
    from skimage import data as skdata
    images = {"cat": Image.fromarray(skdata.chelsea()), "astronaut": Image.fromarray(skdata.astronaut()),
              "coffee": Image.fromarray(skdata.coffee())}
print("Images:", list(images))

In [ ]:
# 21 distinct colors, one per VOC class (0 = background is dark)
PALETTE = plt.get_cmap("tab20")(np.linspace(0, 1, 20))[:, :3]
PALETTE = np.vstack([[0.15, 0.15, 0.2], PALETTE])

def segment(img):
    x = preprocess(img).unsqueeze(0).to(device)       # Step 3: tensor + batch dimension [1, 3, H, W]
    with torch.no_grad():
        out = dl_model(x)["out"]                      # Step 4: scores [1, 21, H, W]
    out = F.interpolate(out, size=img.size[::-1], mode="bilinear", align_corners=False)   # back to photo size
    return out.argmax(1)[0].cpu().numpy()             # Step 5: class id per pixel [H, W]

for name, img in images.items():
    pred = segment(img)
    ids, counts = np.unique(pred, return_counts=True)  # Step 6: which classes are present
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1, 1, 0.45]})
    ax[0].imshow(img); ax[0].set_title("Original image"); ax[0].axis("off")
    ax[1].imshow(img); ax[1].imshow(PALETTE[pred], alpha=0.6); ax[1].set_title("Predicted mask (overlay)"); ax[1].axis("off")
    ax[2].axis("off"); ax[2].set_title("Classes found (% of pixels)")
    for k, (i, c) in enumerate(sorted(zip(ids, counts), key=lambda t: -t[1])):
        ax[2].add_patch(plt.Rectangle((0, 0.9 - k * 0.1), 0.08, 0.07, color=PALETTE[i], transform=ax[2].transAxes))
        ax[2].text(0.12, 0.93 - k * 0.1, f"{categories[i]}  {c / pred.size:.0%}", transform=ax[2].transAxes, va="center", fontsize=11)
    plt.tight_layout(); plt.show()

---
## Data · Oxford-IIIT Pet
Each image has a **trimap**: **1 = pet, 2 = background, 3 = border** (the uncertain outline).
We make a binary mask: **pet + border → 1**, **background → 0**.

> ⚠️ The first version of our script used `trimap == 3` as background. That is the **border**, not the background, so "foreground" became pet + background. See the debugging lesson at the end.

In [ ]:
from torchvision.datasets import OxfordIIITPet
SIZE = (160, 160); N_TRAIN, N_VAL = 400, 100; BATCH = 8; EPOCHS = 5

trainval = OxfordIIITPet("data", split="trainval", target_types="segmentation", download=True)
test     = OxfordIIITPet("data", split="test",     target_types="segmentation", download=True)
print(len(trainval), "trainval images,", len(test), "test images")

def to_tensors(img, trimap):
    img = TF.resize(img, SIZE, interpolation=TF.InterpolationMode.BILINEAR)    # image: smooth resize
    trimap = TF.resize(trimap, SIZE, interpolation=TF.InterpolationMode.NEAREST)  # labels: NEAREST keeps 1/2/3
    x = TF.normalize(TF.to_tensor(img), MEAN, STD)
    t = torch.from_numpy(np.array(trimap)).long()
    y = (t != 2).long()                      # 1 = pet or border, 0 = background
    return x, y

def load_subset(ds, n, seed):
    idx = torch.randperm(len(ds), generator=torch.Generator().manual_seed(seed))[:n].tolist()   # random breeds
    xs, ys = zip(*[to_tensors(*ds[i]) for i in idx])
    return torch.stack(xs), torch.stack(ys)

Xtr, Ytr = load_subset(trainval, N_TRAIN, 0)
Xva, Yva = load_subset(test, N_VAL, 1)
print("train:", tuple(Xtr.shape), tuple(Ytr.shape), "| val:", tuple(Xva.shape))
pet_frac = Yva.float().mean().item()
RESULTS.update(N_TRAIN=str(N_TRAIN), N_VAL=str(N_VAL), EPOCHS=str(EPOCHS), SIZE=f"{SIZE[0]}×{SIZE[1]}",
               PET_FRAC=f"{pet_frac:.0%}", BASE_ACC=f"{1 - pet_frac:.3f}")
print(f"Pet pixels in val: {pet_frac:.1%}.  A model that says 'background' everywhere gets accuracy {1 - pet_frac:.3f} but IoU 0.")

img, tri = test[0]
fig, ax = plt.subplots(1, 3, figsize=(12, 3.8))
ax[0].imshow(img); ax[0].set_title("image")
ax[1].imshow(np.array(tri), cmap="viridis"); ax[1].set_title("trimap: 1 pet · 2 background · 3 border")
ax[2].imshow(np.array(tri) != 2, cmap="gray"); ax[2].set_title("our mask: pet + border = 1")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

---
## Training utilities (shared by all experiments)
One loop for every model: forward → cross-entropy loss → backward → Adam step.
Metrics on the 100 validation images: **pixel accuracy** and **IoU of the pet class** (intersection ÷ union over all pixels).

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xva, Yva), batch_size=BATCH)
SHOW = [0, 1, 2, 3]                       # the same 4 validation images in every figure

@torch.no_grad()
def evaluate(model):
    model.eval(); loss_sum = correct = inter = union = 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        logits = logits_of(model, x)
        loss_sum += F.cross_entropy(logits, y, reduction="sum").item()
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        inter += ((pred == 1) & (y == 1)).sum().item(); union += ((pred == 1) | (y == 1)).sum().item()
    n = Yva.numel()
    return loss_sum / n, correct / n, inter / max(union, 1)

def train(key, title, model, param_groups, frozen=()):
    model = model.to(device)
    opt = torch.optim.Adam(param_groups)
    trainable, total = count_params(model)
    print(f"{title}\n  trainable weights: {trainable/1e6:.2f} M of {total/1e6:.2f} M")
    hist = {"train_loss": [], "val_loss": [], "val_acc": [], "val_iou": []}
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train()
        for m in frozen: m.eval()          # frozen parts: keep their batch-norm statistics fixed too
        run, seen = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            flip = torch.rand(x.size(0), device=device) < 0.5          # augmentation: random horizontal flip
            x = torch.where(flip[:, None, None, None], x.flip(-1), x)
            y = torch.where(flip[:, None, None], y.flip(-1), y)
            opt.zero_grad()
            loss = F.cross_entropy(logits_of(model, x), y)
            loss.backward(); opt.step()
            run += loss.item() * x.size(0); seen += x.size(0)
        vl, va, vi = evaluate(model)
        for k, v in zip(hist, (run / seen, vl, va, vi)): hist[k].append(v)
        print(f"  Epoch {ep+1}/{EPOCHS} | train_loss={run/seen:.4f} | val_loss={vl:.4f} val_acc={va:.4f} val_iou={vi:.4f}")
    secs = time.time() - t0
    K = key.upper()
    RESULTS.update({f"{K}_TRAIN": f"{trainable/1e6:.2f} M", f"{K}_TOTAL": f"{total/1e6:.1f} M",
                    f"{K}_ACC": f"{hist['val_acc'][-1]:.3f}", f"{K}_IOU": f"{hist['val_iou'][-1]:.3f}",
                    f"{K}_TIME": f"{secs:.0f} s"})
    save_figures(key, title, model, hist)
    return model, hist

@torch.no_grad()
def predict(model, idx):
    model.eval(); return logits_of(model, Xva[idx].to(device)).argmax(1).cpu()

def save_figures(key, title, model, hist):
    ep = range(1, EPOCHS + 1)
    fig, ax = plt.subplots(1, 2, figsize=(8, 3.2))
    ax[0].plot(ep, hist["train_loss"], "o-", label="train"); ax[0].plot(ep, hist["val_loss"], "o-", label="val")
    ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(ep, hist["val_acc"], "o-", label="val pixel accuracy"); ax[1].plot(ep, hist["val_iou"], "o-", label="val IoU (pet)")
    ax[1].set_ylim(0, 1); ax[1].set_title("validation metrics"); ax[1].set_xlabel("epoch"); ax[1].legend(loc="lower right")
    for a in ax: a.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"results/{key}_curves.png", dpi=150); plt.show()

    pred = predict(model, SHOW)
    fig, ax = plt.subplots(3, len(SHOW), figsize=(2.3 * len(SHOW), 6.6))
    for j, i in enumerate(SHOW):
        ax[0, j].imshow(denormalize(Xva[i]).permute(1, 2, 0)); ax[1, j].imshow(Yva[i], cmap="gray", vmin=0, vmax=1)
        ax[2, j].imshow(pred[j], cmap="gray", vmin=0, vmax=1)
    for r, t in enumerate(["image", "ground truth", "prediction"]): ax[r, 0].set_ylabel(t, fontsize=12)
    for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.savefig(f"results/{key}_preds.png", dpi=150); plt.show()

MODELS = {}

---
## Experiment 1 · FCN-ResNet50, pretrained backbone **frozen**
Backbone: ResNet-50 with **ImageNet** weights (`weights_backbone=...`). We freeze it and train only the FCN head:
Conv 3×3 (2048 → 512) → BN → ReLU → Dropout → Conv 1×1 (512 → **2** classes).

In [ ]:
from torchvision.models.segmentation import fcn_resnet50
from torchvision.models import ResNet50_Weights

set_seed(42)
m1 = fcn_resnet50(weights=None, weights_backbone=ResNet50_Weights.IMAGENET1K_V1, num_classes=2, aux_loss=False)
for p in m1.backbone.parameters():
    p.requires_grad = False                                   # freeze the encoder
MODELS["e1"], H1 = train("e1", "Exp 1 · FCN-ResNet50, frozen pretrained backbone", m1,
                         [{"params": m1.classifier.parameters(), "lr": 1e-3}], frozen=[m1.backbone])

---
## Experiment 2 · U-Net **from scratch**
Our own U-Net (4 levels, 64 → 1024 channels, padded convs so no cropping). **Nothing is pretrained, everything is trained.**

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class Down(nn.Module):                                  # max pool (size ÷ 2), then two convs
    def __init__(self, cin, cout):
        super().__init__(); self.pool = nn.MaxPool2d(2); self.conv = DoubleConv(cin, cout)
    def forward(self, x): return self.conv(self.pool(x))

class Up(nn.Module):                                    # up-conv (size × 2), concat skip, two convs
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.up = nn.ConvTranspose2d(cin, cin // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(cin // 2 + cskip, cout)
    def forward(self, x, skip):
        x = self.up(x)
        return self.conv(torch.cat([skip, x], dim=1))   # skip connection

class UNet(nn.Module):
    def __init__(self, cin=3, n_classes=2):
        super().__init__()
        self.enc1 = DoubleConv(cin, 64); self.enc2 = Down(64, 128); self.enc3 = Down(128, 256); self.enc4 = Down(256, 512)
        self.bottleneck = Down(512, 1024)
        self.up1 = Up(1024, 512, 512); self.up2 = Up(512, 256, 256); self.up3 = Up(256, 128, 128); self.up4 = Up(128, 64, 64)
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)
    def forward(self, x):
        s1 = self.enc1(x); s2 = self.enc2(s1); s3 = self.enc3(s2); s4 = self.enc4(s3)   # 160, 80, 40, 20
        b = self.bottleneck(s4)                                                        # 10 × 10 × 1024
        x = self.up1(b, s4); x = self.up2(x, s3); x = self.up3(x, s2); x = self.up4(x, s1)
        return self.outc(x)                                                            # [B, 2, 160, 160]

set_seed(42)
m2 = UNet()
MODELS["e2"], H2 = train("e2", "Exp 2 · U-Net from scratch", m2, [{"params": m2.parameters(), "lr": 1e-3}])

---
## Experiment 3 · U-Net + pretrained ResNet34 encoder, **encoder frozen**
`segmentation_models_pytorch` builds the whole U-Net: pretrained encoder + decoder (upsampling + skips) + 1×1 head.

In [ ]:
import segmentation_models_pytorch as smp
set_seed(42)
m3 = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=2)
for p in m3.encoder.parameters():
    p.requires_grad = False                                   # Case 1: freeze encoder
trainable_params = [p for p in m3.parameters() if p.requires_grad]   # decoder + head only
MODELS["e3"], H3 = train("e3", "Exp 3 · U-Net + ResNet34, frozen encoder", m3,
                         [{"params": trainable_params, "lr": 1e-3}], frozen=[m3.encoder])

---
## Experiment 4 · U-Net + ResNet34, **fine-tune everything** with different learning rates
Pretrained encoder: already good → small steps (1e-4). New decoder: starts from random → bigger steps (1e-3).

In [ ]:
set_seed(42)
m4 = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=2)
MODELS["e4"], H4 = train("e4", "Exp 4 · U-Net + ResNet34, fine-tune all", m4, [
    {"params": m4.encoder.parameters(), "lr": 1e-4},                                   # pretrained: slow
    {"params": list(m4.decoder.parameters()) + list(m4.segmentation_head.parameters()), "lr": 1e-3},  # new: fast
])

---
## Comparison: same data, same epochs, four strategies

In [ ]:
names = {"e1": "1 · FCN-R50\nfrozen backbone", "e2": "2 · U-Net\nfrom scratch",
         "e3": "3 · U-Net-R34\nfrozen encoder", "e4": "4 · U-Net-R34\nfine-tune all"}
keys = [k for k in names if k in MODELS]
print(f"{'model':38s} {'trainable':>10s} {'time':>7s} {'pixel acc':>10s} {'IoU':>7s}")
for k in keys:
    K = k.upper()
    print(f"{names[k].replace(chr(10), ' '):38s} {RESULTS[K+'_TRAIN']:>10s} {RESULTS[K+'_TIME']:>7s} {RESULTS[K+'_ACC']:>10s} {RESULTS[K+'_IOU']:>7s}")
print(f"{'baseline: all background':38s} {'0':>10s} {'-':>7s} {RESULTS['BASE_ACC']:>10s} {'0.000':>7s}")

fig, ax = plt.subplots(figsize=(8, 3.4))
xs = np.arange(len(keys)); w = 0.38
acc = [float(RESULTS[k.upper() + "_ACC"]) for k in keys]; iou = [float(RESULTS[k.upper() + "_IOU"]) for k in keys]
b1 = ax.bar(xs - w/2, acc, w, label="pixel accuracy", color="#93C5FD"); b2 = ax.bar(xs + w/2, iou, w, label="IoU (pet)", color="#2563EB")
for b in list(b1) + list(b2): ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01, f"{b.get_height():.2f}", ha="center", fontsize=9)
ax.axhline(float(RESULTS["BASE_ACC"]), ls=":", color="gray"); ax.text(len(keys) - 0.5, float(RESULTS["BASE_ACC"]) + 0.01, "all-background accuracy", ha="right", color="gray", fontsize=9)
ax.set_xticks(xs); ax.set_xticklabels([names[k] for k in keys], fontsize=9); ax.set_ylim(0, 1.08); ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.savefig("results/compare_bars.png", dpi=150); plt.show()

preds = {k: predict(MODELS[k], SHOW) for k in keys}
fig, ax = plt.subplots(len(SHOW), 2 + len(keys), figsize=(2.0 * (2 + len(keys)), 2.0 * len(SHOW)))
for r, i in enumerate(SHOW):
    ax[r, 0].imshow(denormalize(Xva[i]).permute(1, 2, 0)); ax[r, 1].imshow(Yva[i], cmap="gray", vmin=0, vmax=1)
    for c, k in enumerate(keys): ax[r, 2 + c].imshow(preds[k][r], cmap="gray", vmin=0, vmax=1)
for c, t in enumerate(["image", "ground truth"] + [names[k] for k in keys]): ax[0, c].set_title(t, fontsize=9)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig("results/compare_preds.png", dpi=150); plt.show()

---
## Debugging lesson: "val_acc = val_iou = 0.8723 every epoch"
The first version of the FCN script had two bugs:
1. `weights_backbone=None` → the backbone was **random**, then frozen. Random frozen features carry no information.
2. `mask = where(trimap == 3, 0, 1)` → class 1 was **pet + background** (≈ 87 % of pixels).

Result: the model predicts **class 1 for every pixel**. Then accuracy = IoU = share of class-1 pixels, and nothing changes between epochs.
This cell re-creates bug 1 (random frozen backbone) with the correct labels.

In [ ]:
RUN_BUG_DEMO = True
if RUN_BUG_DEMO:
    set_seed(42); saved_epochs = EPOCHS; EPOCHS = 2
    mb = fcn_resnet50(weights=None, weights_backbone=None, num_classes=2, aux_loss=False)   # bug: no pretrained weights
    for p in mb.backbone.parameters(): p.requires_grad = False
    _, HB = train("bug", "Bug demo · random backbone, frozen", mb,
                  [{"params": mb.classifier.parameters(), "lr": 1e-3}], frozen=[mb.backbone])
    EPOCHS = saved_epochs
    pb = predict(mb, SHOW)
    print("Share of pixels predicted 'pet':", f"{pb.float().mean().item():.1%}", "(0% or 100% means it predicts one class everywhere)")
    RESULTS["BUG_ACC"], RESULTS["BUG_IOU"] = f"{HB['val_acc'][-1]:.3f}", f"{HB['val_iou'][-1]:.3f}"